# 04_build_prior_art_universe

**Objective:** Build the prior-art universe by streaming the EPO and USPTO PaECTER embeddings and appending in-house recovered embeddings, deduplicating families with EPO preferred over USPTO.

**Inputs:** EPO/USPTO PaECTER embedding parquets; `family_dates.parquet`; `recovered_embeddings.parquet`; `patent_master.parquet`.

**Outputs:** `prior_art_universe.parquet` (`docdb_family_id`, `lens_id`, `priority_date`, float16 1024-dim `embedding`, `embedding_source`).

In [ ]:
# Imports
from pathlib import Path
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

In [ ]:
# Paths and constants
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
EPO       = RAW / "embeddings" / "EPO_PaECTER_embeddings.parquet"
USP       = RAW / "embeddings" / "USPTO_PaECTER_embeddings.parquet"
DATES     = RAW / "bigquery" / "family_dates.parquet"
RECOVERED = PROC / "recovered_embeddings.parquet"
MASTER    = PROC / "patent_master.parquet"
OUT       = PROC / "prior_art_universe.parquet"

BATCH = 50_000
DIM   = 1024

In [ ]:
# Output parquet schema
schema = pa.schema([
    pa.field("docdb_family_id",  pa.int64()),
    pa.field("lens_id",          pa.string()),
    pa.field("priority_date",    pa.date32()),
    pa.field("embedding",        pa.list_(pa.float16())),
    pa.field("embedding_source", pa.string()),
])

## Date lookup

In [ ]:
# Load family priority dates into a lookup dict
t0 = time.time()
print("Lade Datums-Lookup ...")
dates = pd.read_parquet(DATES, columns=["docdb_family_id","family_priority_date"])
date_dict = dict(zip(
    dates["docdb_family_id"].astype("int64").values,
    dates["family_priority_date"].values,
))
del dates
print(f"  {len(date_dict):,} Eintraege  ({time.time()-t0:.1f}s)")

## Writer and streaming helpers

In [ ]:
# Parquet writer and PaECTER streaming helpers
writer = pq.ParquetWriter(OUT, schema, compression="zstd")
seen = set()
total = 0

def make_table(fam_ids, lens_ids, date_vals, emb_2d_f16, source_label):
    n = len(fam_ids)
    flat = emb_2d_f16.reshape(-1)
    offsets = np.arange(0, (n + 1) * DIM, DIM, dtype=np.int32)
    emb_arr = pa.ListArray.from_arrays(
        pa.array(offsets),
        pa.array(flat, type=pa.float16()),
    )
    return pa.Table.from_arrays([
        pa.array(fam_ids,  type=pa.int64()),
        pa.array(lens_ids if lens_ids is not None else [None]*n, type=pa.string()),
        pa.array(date_vals, type=pa.date32()),
        emb_arr,
        pa.array([source_label]*n, type=pa.string()),
    ], schema=schema)

def stream_paecter(path, source_label, dedup_seen):
    """Streamt eine PaECTER-Datei und schreibt batched ins Output.
    Wenn dedup_seen != None: ueberspringe family_ids, die schon im Set sind."""
    global total
    pf = pq.ParquetFile(path)
    n_total = pf.metadata.num_rows
    pbar = tqdm(total=n_total, desc=f"{source_label:5s}", unit="rows")
    n_skipped = 0
    for batch in pf.iter_batches(batch_size=BATCH, columns=["embedding","docdb_family_id"]):
        fam_ids = batch.column("docdb_family_id").to_numpy().astype("int64")
        n_in = len(fam_ids)

        if dedup_seen is not None:
            mask = np.fromiter((f not in dedup_seen for f in fam_ids),
                               dtype=bool, count=n_in)
            n_skipped += int((~mask).sum())
        else:
            mask = np.ones(n_in, dtype=bool)

        if not mask.any():
            pbar.update(n_in)
            continue

        emb_flat = batch.column("embedding").values.to_numpy(zero_copy_only=False)
        emb_2d   = emb_flat.reshape(n_in, DIM)
        emb_2d_f16 = emb_2d[mask].astype(np.float16)

        fam_filt   = fam_ids[mask]
        date_vals  = [date_dict.get(int(f)) for f in fam_filt]

        seen.update(fam_filt.tolist())

        table = make_table(fam_filt, None, date_vals, emb_2d_f16, source_label)
        writer.write_table(table)
        total += len(fam_filt)
        pbar.update(n_in)
    pbar.close()
    return n_skipped

## Phase 1: EPO

In [ ]:
# Phase 1: stream all EPO families
print("\n=== Phase 1: EPO ===")
stream_paecter(EPO, "EPO", dedup_seen=None)
print(f"  total bisher: {total:,}")

## Phase 2: USPTO

In [ ]:
# Phase 2: stream USPTO families, skipping those already in EPO
print("\n=== Phase 2: USPTO (Dedup gegen EPO) ===")
n_skip = stream_paecter(USP, "USPTO", dedup_seen=seen)
print(f"  uebersprungen (Family schon in EPO): {n_skip:,}")
print(f"  total bisher: {total:,}")

## Phase 3: RECOVERED

In [ ]:
# Phase 3: append recovered embeddings with publication-level priority dates
print("\n=== Phase 3: RECOVERED ===")
rec = pd.read_parquet(RECOVERED)
pm  = pd.read_parquet(MASTER, engine="fastparquet", columns=["lens_id","priority_date"])
pm["priority_date"] = pd.to_datetime(pm["priority_date"], errors="coerce").dt.date
rec = rec.merge(pm, on="lens_id", how="left")

emb_2d_f16 = np.stack(rec["embedding"].values).astype(np.float16)
n = len(rec)
fam_arr  = pa.array([None]*n, type=pa.int64())
lens_arr = pa.array(rec["lens_id"].tolist(), type=pa.string())
date_arr = pa.array(rec["priority_date"].tolist(), type=pa.date32())
flat = emb_2d_f16.reshape(-1)
offsets = np.arange(0, (n+1)*DIM, DIM, dtype=np.int32)
emb_arr = pa.ListArray.from_arrays(pa.array(offsets), pa.array(flat, type=pa.float16()))
src_arr = pa.array(["RECOVERED"]*n, type=pa.string())

table = pa.Table.from_arrays([fam_arr, lens_arr, date_arr, emb_arr, src_arr], schema=schema)
writer.write_table(table)
total += n
print(f"  RECOVERED: {n:,}")

## Close and report

In [ ]:
# Close the writer and report totals
writer.close()
size_gb = OUT.stat().st_size / 1e9
print("\n=== Fertig ===")
print(f"  total rows : {total:,}")
print(f"  output     : {OUT}")
print(f"  size       : {size_gb:.2f} GB")
print(f"  runtime    : {(time.time()-t0)/60:.1f} min")